# عرض توضيحي لحذف المشبك SRA
## — استخراج المعرفة المحددة فعليًا من ماجستير إدارة الأعمال!

يعد هذا الدفتر عرضًا توضيحيًا تفاعليًا للمقالة *[\[AI Romance\] استخراج المعرفة المحددة فعليًا من ماجستير إدارة الأعمال! "حذف التشابك" و"التطهير" الخاص بـ LLM القابل للتبديل السريع](https://qiita.com/JunSuzukiJapan/items/)*.

في SRA (هندسة التوجيه المتشابك)، يمكن إزالة المعرفة والمشابك العصبية غير الضرورية بطريقتين.

| الطريقة | واجهة برمجة التطبيقات | الغرض |
|------|-----|------|
| **الإزالة الجسدية** | `pop_synapses(N)` | إزالة نقاط الاشتباك العصبي المضافة عبر Hot-Swap من الذيل لاستعادة حجم النموذج |
| **صفر واضح (تطهير)** | `clear_synapses([idx])` | تعطيل نقاط الاشتباك العصبي في المؤشرات المتوسطة وتحويلها إلى فتحات مجانية |

سنؤكد أيضًا **فخ تشابه جيب التمام** الذي يحدث أثناء إزالة الصفر وحله عمليًا.

أخيرًا، قمنا بتطبيق `clear_synapses` على نموذج تم تدريبه فعليًا على مهام متعددة وأثبتنا أن **وظيفة المهمة المستهدفة فقط هي التي يتم تدميرها بينما تظل كل مهمة أخرى سليمة تمامًا (عدم النسيان)**.

---
## الجزء الأول: إزالة التشابك العصبي (`pop_synapses`)

قمنا فعليًا بقطع نقاط الاشتباك العصبي التي تم إلحاقها لاحقًا عبر Hot-Swap، بدءًا من الذيل.
تمامًا مثل إلغاء تثبيت مكون إضافي من نظام التشغيل، يمكنك إزالة أجزاء من دماغ الذكاء الاصطناعي فعليًا.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## الجزء الثاني: إزالة الصفر (التطهير) و"فخ تشابه جيب التمام"

إذا أردنا حذف Synapse في فهرس متوسط، فإن إزالته فعليًا ستؤدي إلى تغيير المعرفات.
لذا بدلًا من ذلك، نستبدل الأوزان بـ `0.0` - "صفر واضح".

ومع ذلك، هناك **فخ مرعب** هنا. يصبح تشابه جيب التمام للمتجه الصفري `0.0`،
وإذا كانت نتائج المشابك العصبية الطبيعية سلبية، فإن المشابك العصبية الفارغة تنتهي بدرجة أعلى وتبدأ في استيعاب البيانات** - وهي ظاهرة الثقب الأسود.

يحتوي جهاز توجيه SRA على **قناع `-inf`** مدمج لمنع هذا الفخ. دعونا التحقق من ذلك في الممارسة العملية.

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## الجزء 3: الدليل الوظيفي — `clear_synapses` على نموذج مُدرب

الآن نصل إلى الحدث الرئيسي. في الجزأين 1 و2، تحققنا من "سلوك واجهة برمجة التطبيقات"،
ولكن ما يهم حقًا هو الدليل الوظيفي: **"بعد الإزالة الصفرية، هل يتم فقدان المعرفة المحذوفة فقط بينما تظل كل المعرفة الأخرى سليمة تمامًا؟"**

باستخدام نفس النهج المتبع في دفتر الملاحظات 05 (تجربة الآفة):
1. تدريب المهام المتعددة على `copy` و`reverse`
2. تحديد المشابك العصبية التي يستخدمها `reverse` بشكل متكرر وإزالتها باستخدام `clear_synapses`
3. تحقق من أن `reverse` يصبح غير قابل للحل بينما يستمر `copy` في تسجيل 100% (صفر نسيان)

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. اختبار أداء ما قبل الحذف وتحديد الهدف
نؤكد أنه يمكن حل كل مهمة بشكل صحيح، ونحدد نقاط الاشتباك العصبي الأكثر استخدامًا بواسطة `reverse`.

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. حذف المشبك عبر `clear_synapses`
في دفتر الملاحظات 05 قمنا بتشغيل `block.w1.data[idx].zero_()` يدويًا، ولكن هنا نستخدم **`clear_synapses` API** الرسمي، والذي يطبق أيضًا قناع `-inf` الخاص بجهاز التوجيه، لإجراء حذف آمن.

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


### 3-3. اختبار الأداء بعد الحذف (التحقق من عدم النسيان مطلقًا)

نختبر مرة أخرى مع إزالة بعض نقاط الاشتباك العصبي.

**النتائج المتوقعة:**
- **النسخ**: الدقة محفوظة (تستخدم نقاط اشتباك عصبي مختلفة، لذا فهي سليمة تمامًا)
- **العكس**: تنخفض الدقة (تختفي نقاط الاشتباك العصبي المتخصصة)

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## ملخص

| تجريبي | ما تبين |
|------|----------------------|
| الجزء 1 | الإزالة الفعلية واستعادة الذاكرة عبر `pop_synapses` |
| الجزء 2 | إزالة الصفر عبر `clear_synapses` والتحقق من قناع `-inf` |
| الجزء 3 | `clear_synapses` على نموذج مدرب -> يتم تدمير المهمة المستهدفة فقط بينما تظل المهام الأخرى سليمة |

وبهذا نكون قد حققنا دورة الحياة الكاملة للذكاء الاصطناعي المعياري: **"التدريب -> التكامل (التبديل السريع) -> الحذف (التطهير) -> إعادة استخدام الفتحة (الكتابة الفوقية)"**.